# Accra (GAMA) Urban Expansion Monitoring — Multi-Year Built-Up Area Trend

**No synthetic data anywhere.** Every result reflects live Landsat
observations, or the notebook stops with a clear error.

## What this measures

Not vegetation loss, not flooding — **built-up area growth** across the
Greater Accra Metropolitan Area, using NDBI (Normalized Difference
Built-up Index, Zha 2003): `(SWIR1 - NIR) / (SWIR1 + NIR)`. Urban
surfaces (concrete, asphalt, rooftops) reflect strongly in the
shortwave infrared and weakly in the near-infrared, giving NDBI > 0;
vegetation is the reverse. A **rising** NDBI trend over multiple years
is the signature of urban expansion.

## Study area

The same real, cited Greater Accra Metropolitan Area boundary used in
the flood mapping analysis — ~1,585 km², 25 districts, ~5 million
people (bounding coordinates 5.0908–5.4672°N, -0.6172–0.0828° from a
peer-reviewed land-use study; see that notebook for full sourcing).
Reused directly here for consistency across this project's Accra work.


In [ ]:
from pathlib import Path
import numpy as np

from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.processor import LandsatExtractor, TimeSeriesAnalyzer
from pygeofetch.processing.preprocessor import Preprocessor
from pygeofetch.viz import Plotter, MapViewer

print("Modules loaded")


## 1. Study Area — Greater Accra Metropolitan Area (real, cited boundary)


In [ ]:
import json

# REAL boundary -- loaded from an official Greater Accra shapefile
# (GREATER_ACCRA.shp), replacing the earlier representative
# approximation entirely. The uploaded file had no .shx/.dbf sidecars,
# but its raw geometry was still readable directly; coordinates were in
# UTM Zone 30N (EPSG:32630, the standard zone for Ghana, confirmed by
# reprojecting and checking against known real locations -- Accra
# centre, Tema, Adenta, and Weija all fall inside the result). This is
# the full Greater Accra REGION boundary (broader than the GAMA urban-
# agglomeration estimate used previously, ~3,700 km2 vs ~1,585 km2) --
# Kasoa falls just outside it, consistent with Kasoa sitting in the
# neighbouring Central Region administratively. Simplified from the
# original 4,872 points to 376 (0.03% area change) for practical use.
boundary_geojson_path = Path("./data/boundaries/greater_accra_boundary.geojson")
if not boundary_geojson_path.exists():
    raise RuntimeError(
        f"Real boundary file not found at {boundary_geojson_path}. "
        f"This notebook uses the official Greater Accra shapefile "
        f"directly -- it does not fall back to an approximation. "
        f"Copy greater_accra_boundary.geojson into ./data/boundaries/ "
        f"before running."
    )
gama_geometry = json.loads(boundary_geojson_path.read_text())
gama_boundary_coords = gama_geometry["coordinates"][0]

gama_lons = [c[0] for c in gama_boundary_coords]
gama_lats = [c[1] for c in gama_boundary_coords]
aoi_extent = (min(gama_lons), max(gama_lons), min(gama_lats), max(gama_lats))

output_dir = Path("./data/accra_urban_expansion")
output_dir.mkdir(parents=True, exist_ok=True)
boundary_path = output_dir / "gama_boundary.geojson"
boundary_path.write_text(json.dumps({
    "type": "FeatureCollection",
    "features": [{"type": "Feature", "properties": {"name": "Greater Accra Metropolitan Area"}, "geometry": gama_geometry}],
}))
print(f"Study area: Greater Accra Metropolitan Area, extent {aoi_extent}")


## 2. Search — multi-year Landsat time series


In [ ]:
client = PyGeoFetch(log_level="INFO")
USGS_USERNAME = "your_ers_username"
USGS_APP_TOKEN = "your_application_token"
client.add_credentials("usgs", username=USGS_USERNAME, api_key=USGS_APP_TOKEN)

date_windows = [
    ("2016-11-01", "2017-02-28"), ("2018-11-01", "2019-02-28"),
    ("2020-11-01", "2021-02-28"), ("2022-11-01", "2023-02-28"),
    ("2023-11-01", "2024-02-28"),
]

scenes = {}
for start, end in date_windows:
    query = SearchQuery(
        geometry=gama_geometry, start_date=start, end_date=end,
        satellites=["Landsat-8", "Landsat-9"], cloud_cover_max=20, max_results=5,
    )
    results = client.search(query, providers=["usgs"])
    results = [r for r in results if r.datetime is not None]
    if results:
        best = min(results, key=lambda r: r.cloud_cover or 100)
        scenes[str(best.datetime.date())] = best
        print(f"  {start[:4]}: {best.id}  ({best.datetime.date()}, {best.cloud_cover:.0f}% cloud)")
    else:
        print(f"  {start[:4]}: no usable scenes found")

if len(scenes) < 3:
    raise RuntimeError(f"Only {len(scenes)} usable date(s) found -- need at least 3. No synthetic fallback used.")
print(f"\n{len(scenes)} dates found -- proceeding with real data only.")


## 3. Download all dates


In [ ]:
download_results = {}
options = DownloadOptions(parallel=1, resume=True, on_failure="skip")

for date_str, scene in scenes.items():
    scene_dir = output_dir / date_str
    results_dl = client.download([scene], destination=scene_dir, options=options)
    r = results_dl[0]
    print(f"  {date_str}: {'OK' if r.success else f'FAILED: {r.error}'}")
    if r.success:
        download_results[date_str] = r

if len(download_results) < 3:
    raise RuntimeError(f"Only {len(download_results)} download(s) succeeded -- need at least 3.")


## 4. Clip-first processing: extract → clip → scale — NIR and SWIR1 (not RED)

NDBI needs NIR and SWIR1, not RED — different bands than the
vegetation-trend notebooks in this project. Correct OLI (Landsat 8/9)
mapping: NIR = SR_B5, SWIR1 = SR_B6. TM/ETM+ (Landsat 4-7): NIR =
SR_B4, SWIR1 = SR_B5. Sensor auto-detected, correct map applied
automatically by `LandsatExtractor`.


In [ ]:
extractor = LandsatExtractor()
pp = Preprocessor()

def process_scene_clip_first_ndbi(source, work_dir, aoi_geometry):
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    individual = extractor._resolve_individual_files(source)
    if individual is not None:
        extracted = individual
        sensor_hint = next(iter(extracted.keys()), "")
    else:
        bundle_path = extractor._resolve_path(source)
        if bundle_path is None:
            return None
        extracted = extractor.extract_bundle(bundle_path, work_dir / "extracted")
        sensor_hint = bundle_path.name

    sensor = extractor._detect_sensor(sensor_hint)
    band_map = {
        "OLI": {"nir": "SR_B5", "swir1": "SR_B6"},
        "TM":  {"nir": "SR_B4", "swir1": "SR_B5"},
    }[sensor]

    clipped_paths = {}
    for band_name in ["nir", "swir1", "qa_pixel"]:
        suffix = band_map.get(band_name) if band_name in band_map else "QA_PIXEL"
        raw_path = extractor.find_band(extracted, suffix)
        if raw_path is None:
            continue
        clip_result = pp.clip(raw_path, geometry=aoi_geometry, output=str(work_dir / f"{band_name}_clipped.tif"))
        if clip_result.success:
            clipped_paths[band_name] = clip_result.output_path

    if "nir" not in clipped_paths or "swir1" not in clipped_paths:
        return None

    band_arrays, profile = {}, None
    for band_name in ["nir", "swir1"]:
        arr, prof = extractor.load_scaled_band(clipped_paths[band_name])
        if profile is None:
            profile = prof
        band_arrays[band_name.upper()] = arr

    if "qa_pixel" in clipped_paths:
        mask = extractor.cloud_mask(clipped_paths["qa_pixel"], shape=next(iter(band_arrays.values())).shape)
        for k in band_arrays:
            band_arrays[k] = np.where(mask, np.nan, band_arrays[k])

    return band_arrays, profile, sensor


date_bands = {}
clip_profile = None
for date_str, result in download_results.items():
    processed = process_scene_clip_first_ndbi(result, output_dir / date_str, gama_geometry)
    if processed is None:
        print(f"  {date_str}: could not process (missing NIR/SWIR1)")
        continue
    band_arrays, profile, sensor = processed
    clip_profile = clip_profile or profile
    date_bands[date_str] = {}

    import rasterio
    for band_key, arr in band_arrays.items():
        out_path = output_dir / date_str / f"{band_key.lower()}_scaled_clipped.tif"
        prof = profile.copy()
        prof.update(dtype="float32", count=1)
        with rasterio.open(out_path, "w", **prof) as dst:
            dst.write(arr.astype(np.float32)[np.newaxis, :, :])
        date_bands[date_str][band_key] = out_path

    print(f"  {date_str}: clipped + scaled + masked ({sensor})")

if len(date_bands) < 3:
    raise RuntimeError(f"Only {len(date_bands)} date(s) processed -- need at least 3.")


## 5. Build the NDBI time series and compute the urbanization trend


In [ ]:
ts = TimeSeriesAnalyzer(index="NDBI")
stack = ts.build_index_stack(date_bands)
print(stack)

trend = ts.trend(stack)
print(f"\nTrend range: [{np.nanmin(trend):.4f}, {np.nanmax(trend):.4f}] NDBI/year")
print(f"Mean trend: {np.nanmean(trend):.4f} NDBI/year")

expanding_pct = 100 * np.nanmean(trend > 0.02)
print(f"Area with clear rising NDBI (> 0.02/year, likely new urban development): {expanding_pct:.1f}%")


## 6. Built-up area change — first date vs. last date


In [ ]:
first_date, last_date = stack.dates[0], stack.dates[-1]
first_ndbi = stack.values[0]
last_ndbi = stack.values[-1]

URBAN_THRESHOLD = 0.0  # NDBI > 0 = built-up, per Zha 2003
first_urban_pct = 100 * np.nanmean(first_ndbi > URBAN_THRESHOLD)
last_urban_pct = 100 * np.nanmean(last_ndbi > URBAN_THRESHOLD)

print(f"Built-up area, {first_date}: {first_urban_pct:.1f}% of GAMA")
print(f"Built-up area, {last_date}: {last_urban_pct:.1f}% of GAMA")
print(f"Net change: {last_urban_pct - first_urban_pct:+.1f} percentage points over {len(stack.dates)} observed dates")


## 7. Visualization — growth intensity, distinct from the deforestation color scheme


In [ ]:
pl = Plotter(figsize=(12, 9))

pl.plot_raster(
    trend, title="Accra (GAMA) — Urban Expansion Trend (NDBI/year)",
    colormap="YlOrRd", vmin=0, vmax=0.05, extent=aoi_extent,
    colorbar_label="NDBI / year (darker = faster urbanization)",
    output=str(output_dir / "urban_expansion_trend.png"),
)


In [ ]:
GROWTH_LABELS = {0: "No significant growth", 1: "Moderate new development", 2: "Rapid new development"}
GROWTH_COLORS = {0: "#f0f0f0", 1: "#fd8d3c", 2: "#bd0026"}

def classify_growth(trend_arr):
    codes = np.zeros(trend_arr.shape, dtype=np.int16)
    codes[trend_arr > 0.015] = 1
    codes[trend_arr > 0.035] = 2
    codes[np.isnan(trend_arr)] = 0
    return codes

growth_classified = classify_growth(trend)
pl.plot_classification(
    growth_classified, class_labels=GROWTH_LABELS, class_colors=GROWTH_COLORS,
    title="Accra (GAMA) — New Development Intensity",
    extent=aoi_extent, output=str(output_dir / "urban_growth_classification.png"),
)


## 8. Composite summary figure


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

im0 = axes[0].imshow(first_ndbi, cmap="RdYlGn_r", vmin=-0.3, vmax=0.3, extent=aoi_extent, aspect="auto")
axes[0].set_title(f"NDBI — {first_date}", fontsize=13)
plt.colorbar(im0, ax=axes[0], fraction=0.04)

im1 = axes[1].imshow(last_ndbi, cmap="RdYlGn_r", vmin=-0.3, vmax=0.3, extent=aoi_extent, aspect="auto")
axes[1].set_title(f"NDBI — {last_date}", fontsize=13)
plt.colorbar(im1, ax=axes[1], fraction=0.04)

cmap_growth = ListedColormap([GROWTH_COLORS[i] for i in range(3)])
axes[2].imshow(growth_classified, cmap=cmap_growth, extent=aoi_extent, aspect="auto")
axes[2].set_title(f"Growth Intensity ({last_urban_pct-first_urban_pct:+.1f} pts net)", fontsize=13)

fig.suptitle(f"Accra (GAMA) Urban Expansion — {first_date} to {last_date}", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(output_dir / "urban_expansion_summary.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Interactive map


In [ ]:
from pygeofetch.processing.base import _safe_write_band

mv = MapViewer(center=(5.28, -0.28), zoom=10)
mv.add_vector(str(boundary_path), layer_name="Greater Accra Metropolitan Area",
              style={"color": "#1a1a1a", "weight": 2, "fillOpacity": 0.0})
trend_tif = output_dir / "trend.tif"
_safe_write_band(trend, clip_profile, trend_tif)
mv.add_raster(str(trend_tif), colormap="YlOrRd", vmin=0, vmax=0.05, layer_name="Urban Expansion Trend", opacity=0.75)
mv.add_basemap("SATELLITE")
map_path = output_dir / "interactive_map.html"
mv.save(map_path)
print(f"Interactive map saved: {map_path}")


## 10. Save GIS-ready outputs


In [ ]:
results_dir = output_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
_safe_write_band(trend, clip_profile, results_dir / "ndbi_trend.tif")
_safe_write_band(growth_classified.astype(np.float32), clip_profile, results_dir / "growth_classification.tif")
print(f"Saved to: {results_dir}")
for f in sorted(results_dir.glob("*.tif")):
    print(f"  {f.name}")


## Summary

- **Real, multi-year NDBI trend** across the full Greater Accra
  Metropolitan Area (~1,585 km²), not a two-date snapshot
- **No synthetic data** — every figure reflects live Landsat
  observations, or the notebook stops with a clear error
- Net built-up area change and a spatial growth-intensity map, both
  saved as GIS-ready GeoTIFFs

### Limitations worth being direct about

NDBI can be confused by bare soil and rock outcrops, which share
built-up surfaces' spectral signature — a rising NDBI trend in a
specific area is not automatically confirmed new construction without
visual or ground verification. Cloud cover and seasonal vegetation
differences between the dry-season windows used here can also
introduce noise; the multi-year trend fit (rather than a single
before/after comparison) is what makes this reasonably robust to that,
not immune to it.
